In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

from seqAE_model import SeqAutoencoder
from contra_seq_dataset import AnchoredSampler
from torch.utils.data import DataLoader, RandomSampler
from contra_seq_dataset import ContraSeqDataset, get_dataset_array, get_anc_map
from losses import SupConLoss, padce_loss
from tqdm.notebook import tqdm
import torch
import random

In [2]:
## Dataset

anc_path = 'model_dataset/anchor_smiles.csv'
aug_path = 'model_dataset/augmented_smiles_balanced.csv'

ds = ContraSeqDataset(anc_path, aug_path)
ds_arr = get_dataset_array(anc_path, aug_path)
anc_map = get_anc_map(ds_arr)

In [ ]:
import torch.nn as nn
import pandas as pd
import matplotlib.pylab as plt

# torch.cuda.empty_cache()
use_cuda = False
device =  torch.device("cuda" if use_cuda else "cpu")
supcon_loss = SupConLoss()
lr = 0.00001

model = SeqAutoencoder(n_tokens = ds.n_tokens, max_len = 122,
                       dim_emb=512, heads=8, dim_hidden=32,
                       L_enc=6, L_dec=6, dim_ff=2048, 
                       drpt=0.1, actv='relu', eps=0.6, b_first=True)

p = 'models/supconrecon_20220412a/checkpoint_supconrecon_20220412a_49'
model.load_state_dict(torch.load(p), strict = False)

if use_cuda==True and torch.cuda.device_count() > 1:
    print("Let's use", torch.cuda.device_count(), "GPUs!")
    model = nn.DataParallel(model)
    model.to(device)
else:
    model = model.to(device)
    
optimizer = torch.optim.Adam(model.parameters(), lr = lr)    
model.eval()

In [ ]:
import seaborn as sns
losses = pd.read_csv('training_logs/losses_20220412a',usecols=['SupCon','Recon'])

BATCH_SIZE = 32

figsize=(10,5)
plt.figure(figsize=figsize)
for label,data in losses.iteritems():
    plt.plot(data, label=label)

plt.xlim(-100,5000)
plt.title(f'Batch size: {BATCH_SIZE}')
plt.grid(True)
plt.xlabel('Run')
plt.legend(loc='upper right')
plt.show()
        
# sns.lineplot(losses, y=losses.index)

## UMAP a sample of training data.

In [ ]:
from rdkit.Chem import AllChem as Chem
from rdkit.Chem import Draw
from matplotlib.offsetbox import OffsetImage,AnnotationBbox
import numpy as np

import copy
_df = copy.deepcopy(ds_arr)
# df.columns = [x[0].upper() + x[1:] for x in df.columns]
_df.columns = ['Smiles','Atype','Label']
# _df

num_to_eval = 100
rand = random.sample(range(0,10000),num_to_eval)
# print(rand)
# print([anc_map[x] for x in rand])
rand_idc = np.concatenate([anc_map[x] for x in rand],axis=0)
# print(rand_idc)
# print(rand_idc.shape)
df = _df.iloc[rand_idc]
display(df)

from rdkit.Chem import PandasTools

PandasTools.AddMoleculeColumnToFrame(df,'Smiles','Mol',includeFingerprints=False)
df = df[["Smiles","Mol","Label","Atype"]]
# display(df)

### Get latent code from model.

In [ ]:
sampler = AnchoredSampler(rand, anc_map, num_to_eval, drop_last = False)
loader = DataLoader(ds, batch_sampler=sampler, num_workers=0, pin_memory=True)

for samp in loader:
    for k,v in samp.items():
        if torch.is_tensor(v):
            samp[k] = v.to(device)
    latent, _ = model.forward(samp['seq'], samp['pad_mask'], 
                                    samp['avg_mask'], samp['out_mask'])
    latent = torch.nn.functional.normalize(latent, p=2.0, dim=1)
latent = latent.detach().numpy()
print(latent.shape)

In [ ]:
import umap.umap_ as umap
umapper = umap.UMAP(n_neighbors=15, min_dist=0.05, n_components=2, metric='euclidean')
embedding = umapper.fit_transform(latent)

df['x'] = embedding[:, 0]
df['y'] = embedding[:, 1]

In [ ]:
%matplotlib inline
import plotly.graph_objs as go

from ipywidgets import Image, Layout, HBox, VBox
from rdkit.Chem import Draw
import PIL

import ipywidgets as widgets
import io

df_anchors = df[df.Atype=='Anc']
df_augmentations = df[df.Atype=='Aug']

# Assemble all the traces.
trace_anchors = (go.Scattergl(x=df_anchors['x'], 
                           y=df_anchors['y'], 
                           name='Substrate',
                           marker=dict(size=3, opacity=0.25, color='limegreen'),
                           hoverinfo='text',
                           text=df_anchors['Smiles'],
                           mode='markers'))
trace_augmentations = (go.Scattergl(x=df_augmentations['x'], 
                           y=df_augmentations['y'], 
                           name='Non-substrate',
                           marker=dict(size=3, opacity=0.25, color='crimson'), #chartreuse
                           hoverinfo='text',
                           text=df_augmentations['Smiles'],
                           mode='markers'))
data = [trace_augmentations, trace_anchors, ]

# Make figure.
fig = go.FigureWidget(data=data)
# fig.layout.hovermode = 'closest'
fig.update_layout(template='simple_white', autosize=False, height=512)
fig.update_yaxes(scaleanchor = "x", scaleratio = 1.)

# Show-mol-on-hover function.
img = open('white.png','rb').read()
# img = PIL.Image.new('1', (256, 256)).tobytes()
# image_widget = Image(value=img)

def hover_fn(trace, points, state):
    
    if points.point_inds:
        if trace['name']=='Substrate':
            my_df = df_anchors
        elif trace['name']=='Non-substrate':
            my_df = df_augmentations
            
        idx = points.point_inds[0]
        datum = my_df.iloc[idx]
        mol = datum.Mol

        img = Draw.MolToImage(mol,size=(256, 512))  ### to make mol image smaller, alter first dim. 
        imgByteArr = io.BytesIO()
        img.save(imgByteArr, format='PNG')
        imgByteArr = imgByteArr.getvalue()
        image_widget.value = imgByteArr

for trace_num in range(len(fig.data)):
    fig.data[trace_num].on_hover(hover_fn)  
HBox([HBox([image_widget]),fig])